# Wan2.1-I2V Animal Video Generator
Auto-generated by viral-animal-video pipeline.
Generates 6-8 overlapping video segments and blends them into a continuous shot.

In [ ]:
# ── AUTO-INJECTED CONFIG (overwritten by pipeline) ────────────────────────
VIDEO_CONFIG = {
  "title": "Test Video",
  "camera_style": "CCTV fisheye",
  "duration": 40,
  "video_prompt": "A golden retriever sitting by a door, shot from a CCTV fisheye camera mounted in the upper corner of a room. Grainy footage, timestamp overlay, dim hallway lighting, dog looking at door with ears perked, ultra-realistic, emotional atmosphere.",
  "animal_type": "golden retriever",
  "scenario": "waiting at the door every day for months",
  "emotion_progression": ["anticipation", "quiet hope", "recognition", "overwhelming joy", "peaceful reunion"]
}
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
# ── INSTALL DEPENDENCIES ──────────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

pip_install(
    'diffusers>=0.27.0',
    'transformers>=4.40.0',
    'accelerate',
    'imageio[ffmpeg]',
    'Pillow',
    'torch',
    'torchvision',
    'numpy',
    'ffmpeg-python',
)
print('Dependencies installed.')

In [ ]:
# ── IMPORTS & SETTINGS ────────────────────────────────────────────────────
import os, gc, json, math, random, shutil
import numpy as np
import torch
from pathlib import Path
from PIL import Image
import imageio

# Wan2.1 via diffusers
from diffusers import WanVideoToVideoPipeline
from diffusers.utils import export_to_video

# Constants
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE           = torch.float16 if DEVICE == 'cuda' else torch.float32
MODEL_ID        = 'Wan-AI/Wan2.1-I2V-1.3B'
WIDTH           = 480
HEIGHT          = 854   # 9:16
FPS             = 8
FRAMES_PER_SEG  = 48    # ~6s at 8fps
OVERLAP_FRAMES  = 8     # frames to crossfade between segments
INFERENCE_STEPS = 30
GUIDANCE_SCALE  = 7.5
OUTPUT_DIR      = '/kaggle/working'

TARGET_DURATION = VIDEO_CONFIG.get('duration', 40)
TARGET_FRAMES   = TARGET_DURATION * FPS  # e.g. 40s × 8fps = 320 frames
# Segments needed = ceil((target - overlap) / (frames_per_seg - overlap))
NUM_SEGMENTS = math.ceil(
    (TARGET_FRAMES - OVERLAP_FRAMES) / (FRAMES_PER_SEG - OVERLAP_FRAMES)
)
NUM_SEGMENTS = max(6, min(8, NUM_SEGMENTS))

print(f'Device: {DEVICE} | Dtype: {DTYPE}')
print(f'Target: {TARGET_DURATION}s / {TARGET_FRAMES} frames')
print(f'Segments: {NUM_SEGMENTS}')

In [ ]:
# ── LOAD MODEL ────────────────────────────────────────────────────────────
print('Loading Wan2.1-I2V-1.3B…')

pipe = WanVideoToVideoPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)
pipe.enable_model_cpu_offload()  # T4 VRAM optimization
pipe.enable_vae_slicing()         # further VRAM savings

if hasattr(pipe, 'enable_xformers_memory_efficient_attention'):
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception:
        pass

print('Model loaded.')

In [ ]:
# ── PROMPT BUILDER ────────────────────────────────────────────────────────

def build_segment_prompt(base_prompt: str, emotion: str, segment_idx: int, total: int) -> str:
    """
    Modulate prompt based on segment position for emotional progression.
    """
    position = segment_idx / max(1, total - 1)  # 0.0 → 1.0

    if position < 0.2:
        pacing = 'slow, still, quiet, anticipation'
    elif position < 0.5:
        pacing = 'slight movement, growing tension, raw emotion'
    elif position < 0.8:
        pacing = 'emotional peak, intense movement, heartbreaking'
    else:
        pacing = 'resolution, soft movement, emotional payoff'

    return (
        f"{base_prompt}. "
        f"Emotional tone: {emotion}. "
        f"Pacing: {pacing}. "
        f"Ultra-realistic, photorealistic, cinematic. "
        f"No text overlays in video itself."
    )


def get_negative_prompt() -> str:
    return (
        'cartoon, anime, illustration, painting, sketch, watermark, text, '
        'low quality, blurry motion, jerky motion, cut, jump cut, '
        'multiple animals unless specified, human face close-up, '
        'unnatural movement'
    )

print('Prompt builder ready.')

In [ ]:
# ── SEGMENT GENERATOR ─────────────────────────────────────────────────────

def frames_to_pil(frames_tensor) -> list:
    """Convert tensor output to list of PIL Images."""
    if isinstance(frames_tensor, torch.Tensor):
        # (T, H, W, C) or (T, C, H, W)
        if frames_tensor.shape[-1] == 3:
            arr = frames_tensor.cpu().numpy()
        else:
            arr = frames_tensor.permute(0, 2, 3, 1).cpu().numpy()
        arr = (arr * 255).clip(0, 255).astype(np.uint8)
        return [Image.fromarray(f) for f in arr]
    # already list of PIL
    return frames_tensor


def generate_segment(
    prompt: str,
    negative_prompt: str,
    init_image: Image.Image,
    seed: int,
) -> list:
    """Generate FRAMES_PER_SEG frames from an init image."""
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    with torch.inference_mode():
        output = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            image=init_image,
            num_frames=FRAMES_PER_SEG,
            width=WIDTH,
            height=HEIGHT,
            num_inference_steps=INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator,
        )

    frames = output.frames if hasattr(output, 'frames') else output[0]
    if not isinstance(frames[0], Image.Image):
        frames = frames_to_pil(frames)

    # Free cache after each segment
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    return frames


print('Segment generator ready.')

In [ ]:
# ── CROSSFADE BLENDER ─────────────────────────────────────────────────────

def blend_overlap(frames_a: list, frames_b: list, overlap: int) -> list:
    """
    Blend the last `overlap` frames of frames_a with the first `overlap`
    frames of frames_b using a linear crossfade.
    Returns frames_a[:-overlap] + blended_overlap + frames_b[overlap:]
    """
    if overlap == 0:
        return frames_a + frames_b

    tail = frames_a[-overlap:]
    head = frames_b[:overlap]

    blended = []
    for i, (fa, fb) in enumerate(zip(tail, head)):
        alpha = (i + 1) / (overlap + 1)  # 0→1
        fa_arr = np.array(fa, dtype=np.float32)
        fb_arr = np.array(fb, dtype=np.float32)
        blend_arr = ((1 - alpha) * fa_arr + alpha * fb_arr).clip(0, 255).astype(np.uint8)
        blended.append(Image.fromarray(blend_arr))

    return frames_a[:-overlap] + blended + frames_b[overlap:]


print('Crossfade blender ready.')

In [ ]:
# ── INITIAL FRAME GENERATOR ───────────────────────────────────────────────
# Generate a single seed frame via text→image (using pipe's text encoder + VAE)
# We use a simple PIL placeholder since Wan2.1 I2V needs an init image.

def create_init_frame(prompt: str, seed: int) -> Image.Image:
    """
    Create an initial seed frame.
    Uses StableDiffusion-style text-to-image if available,
    otherwise creates a neutral gray placeholder.
    The I2V model will override the content anyway after the first frame.
    """
    try:
        from diffusers import StableDiffusionPipeline
        # Quick 1-step latent for init
        sdpipe = StableDiffusionPipeline.from_pretrained(
            'runwayml/stable-diffusion-v1-5',
            torch_dtype=DTYPE,
        ).to(DEVICE)
        sdpipe.safety_checker = None
        gen = torch.Generator(device=DEVICE).manual_seed(seed)
        img = sdpipe(
            prompt, height=HEIGHT, width=WIDTH,
            num_inference_steps=20,
            guidance_scale=7.5,
            generator=gen,
        ).images[0]
        del sdpipe
        gc.collect()
        torch.cuda.empty_cache()
        return img
    except Exception as e:
        print(f'SD init frame failed ({e}), using noise placeholder')
        rng = np.random.default_rng(seed)
        noise = rng.integers(40, 80, (HEIGHT, WIDTH, 3), dtype=np.uint8)
        return Image.fromarray(noise)


print('Init frame generator ready.')

In [ ]:
# ── MAIN VIDEO GENERATION LOOP ────────────────────────────────────────────
import time

base_prompt     = VIDEO_CONFIG['video_prompt']
emotions        = VIDEO_CONFIG.get('emotion_progression', ['neutral'] * 5)
negative_prompt = get_negative_prompt()
base_seed       = random.randint(0, 2**31)

print(f'\nStarting generation: {NUM_SEGMENTS} segments')
print(f'Base seed: {base_seed}')
print(f'Prompt: {base_prompt[:120]}…\n')

all_segments: list = []   # list of frame lists

for seg_idx in range(NUM_SEGMENTS):
    t_start = time.time()

    # Pick emotion for this segment
    emotion_idx = min(seg_idx, len(emotions) - 1)
    emotion = emotions[emotion_idx]

    # Build prompt for this segment
    seg_prompt = build_segment_prompt(base_prompt, emotion, seg_idx, NUM_SEGMENTS)

    # Determine init image
    if seg_idx == 0:
        init_img = create_init_frame(seg_prompt, base_seed)
    else:
        # Use last frame of previous segment as init
        init_img = all_segments[-1][-1]

    seg_seed = base_seed + seg_idx * 137  # deterministic but varied

    print(f'[Seg {seg_idx+1}/{NUM_SEGMENTS}] Generating… (emotion: {emotion})')
    frames = generate_segment(seg_prompt, negative_prompt, init_img, seg_seed)

    elapsed = time.time() - t_start
    print(f'[Seg {seg_idx+1}/{NUM_SEGMENTS}] Done: {len(frames)} frames in {elapsed:.1f}s')

    all_segments.append(frames)

print(f'\nAll {NUM_SEGMENTS} segments generated.')

In [ ]:
# ── BLEND SEGMENTS ────────────────────────────────────────────────────────
print('Blending segments with crossfade…')

final_frames = all_segments[0]
for i in range(1, len(all_segments)):
    final_frames = blend_overlap(final_frames, all_segments[i], OVERLAP_FRAMES)

# Trim to exact target frame count
final_frames = final_frames[:TARGET_FRAMES]
print(f'Final frame count: {len(final_frames)} (target {TARGET_FRAMES})')

In [ ]:
# ── EXPORT MP4 ────────────────────────────────────────────────────────────
import subprocess as sp

output_path = os.path.join(OUTPUT_DIR, 'raw_video.mp4')

print(f'Exporting {len(final_frames)} frames to {output_path}…')

# Write frames as images first
frames_dir = os.path.join(OUTPUT_DIR, 'frames')
os.makedirs(frames_dir, exist_ok=True)
for i, frame in enumerate(final_frames):
    frame.save(os.path.join(frames_dir, f'frame_{i:05d}.png'))

# Encode with FFmpeg for best compatibility
cmd = [
    'ffmpeg', '-y',
    '-framerate', str(FPS),
    '-i', os.path.join(frames_dir, 'frame_%05d.png'),
    '-c:v', 'libx264',
    '-preset', 'fast',
    '-crf', '20',
    '-pix_fmt', 'yuv420p',
    '-movflags', '+faststart',
    output_path,
]
result = sp.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print('FFmpeg stderr:', result.stderr[-1000:])
    raise RuntimeError('FFmpeg encoding failed')

# Cleanup frame PNGs
shutil.rmtree(frames_dir, ignore_errors=True)

size_mb = os.path.getsize(output_path) / 1_048_576
print(f'\n✅ Export complete: {output_path} ({size_mb:.1f} MB)')
print(f'Duration: {len(final_frames)/FPS:.1f}s')

In [ ]:
# ── VERIFY OUTPUT ─────────────────────────────────────────────────────────
import json as _json

probe_cmd = [
    'ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_streams',
    output_path
]
probe_result = sp.run(probe_cmd, capture_output=True, text=True)
probe_data = _json.loads(probe_result.stdout)

for stream in probe_data.get('streams', []):
    if stream.get('codec_type') == 'video':
        print('Video stream verified:')
        print(f"  Codec: {stream.get('codec_name')}")
        print(f"  Size: {stream.get('width')}x{stream.get('height')}")
        print(f"  FPS: {stream.get('r_frame_rate')}")
        print(f"  Duration: {stream.get('duration', 'N/A')}s")

print('\n🎬 Kaggle notebook complete. Output:', output_path)